In [4]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_fscore_support

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from google.colab import drive
import os
drive.mount('/content/drive')

# ============================== CONFIGURE ME ==============================
# ROOT = the folder on your Google Drive that holds a copy of this repository's
# data: data/nab_cleaned/ (output of scripts/clean_nab.py) and
# data/NAB/labels/combined_windows.json. ae_performance.csv is written to ROOT;
# copy it back to the repository root for meta_learner.py.
ROOT = Path("/content/drive/MyDrive/Final_MetaLearningProject")
# ==========================================================================
NAB_ROOT = ROOT / "data" / "NAB"
DATA_DIR = ROOT / "data" / "nab_cleaned"
LABELS_PATH = NAB_ROOT / "labels" / "combined_windows.json"

# Load all cleaned time-series CSV datasets
csv_files = list(DATA_DIR.glob("**/*.csv"))

datasets = {}
for f in csv_files:
    rel = f.relative_to(DATA_DIR)
    key = rel.with_suffix("")
    key = str(key).replace("\\", "/")
    df = pd.read_csv(f)
    datasets[key] = df

print("Loaded datasets:", len(datasets))

# Load anomaly windows
with open(LABELS_PATH, "r") as f:
    labels = json.load(f)

print("Labelled files:", len(labels))

def apply_labels(df: pd.DataFrame, key: str, labels: dict) -> pd.DataFrame:
    df = df.copy()

    # Ensure timestamp is datetime
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    else:
        df.index = pd.to_datetime(df.index)
        df["timestamp"] = df.index

    df["label"] = 0 # initialize all pts target to 0

    label_key = key if key.endswith(".csv") else key + ".csv"
    windows = labels.get(label_key, [])

    if not windows:
        return df

    #
    for w in windows:
        if isinstance(w, dict):
            start = pd.to_datetime(w.get("start") or w.get("begin") or w.get("start_time"))
            end = pd.to_datetime(w.get("end") or w.get("stop") or w.get("end_time"))
        else:
            start = pd.to_datetime(w[0])
            end = pd.to_datetime(w[1])

        mask = (df["timestamp"] >= start) & (df["timestamp"] <= end)
        df.loc[mask, "label"] = 1 # mark anomalies as 1

    return df

# Build processed dataset dictionary
processed = {}
for key, df in datasets.items():
    processed[key] = apply_labels(df, key, labels)

print("Processed datasets:", len(processed))


## Evaluation helper ##
def scores_to_binary_predictions(scores: pd.Series, top_frac: float = 0.01) -> pd.Series:
    threshold = scores.quantile(1 - top_frac)
    y_pred = (scores >= threshold).astype(int)
    y_pred.name = "y_pred"
    return y_pred

def evaluate_isolation_forest(
    iso_scores: dict,
    processed: dict,
    top_frac: float = 0.01,
):
    rows = []

    for key, scores in iso_scores.items():
        if key not in processed:
            continue

        df = processed[key]
        if "label" not in df.columns:
            continue

        y_true = df["label"].astype(int)
        y_true = y_true.reindex(scores.index).fillna(0).astype(int)
        y_pred = scores_to_binary_predictions(scores, top_frac=top_frac)

        precision, recall, f1, support = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        )

        rows.append({
            "dataset": key,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": int(y_true.sum()),
        })

    if not rows:
        return pd.DataFrame(columns=["precision", "recall", "f1", "support"])

    perf_df = pd.DataFrame(rows).set_index("dataset")
    return perf_df


## Autoencoder model + training ##
class TimeSeriesAutoencoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 32, bottleneck_dim: int = 8):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon

def fit_autoencoder_for_dataset(
    df: pd.DataFrame,
    value_col: str = "value",
    window_size: int = 20,
    hidden_dim: int = 32,
    bottleneck_dim: int = 8,
    num_epochs: int = 10,
    batch_size: int = 128,
    lr: float = 1e-3,
    dataset_name=None,
    verbose: bool = False,
):
    values = df[value_col].to_numpy(dtype=float)
    n = len(values)
    if n < window_size:
        scores = pd.Series(
            np.zeros(n),
            index=df.index,
            name="ae_recon_error",
        )
        return scores, None, None

    # Standardize the series
    scaler = StandardScaler()
    values_scaled = scaler.fit_transform(values.reshape(-1, 1)).ravel()

    # Build sliding-window matrix
    windows = []
    for i in range(n - window_size + 1):
        windows.append(values_scaled[i : i + window_size])
    X = np.array(windows, dtype=np.float32)

    # Torch dataset / dataloader
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dataset = TensorDataset(torch.from_numpy(X))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Model, loss, optimizer
    input_dim = window_size
    model = TimeSeriesAutoencoder(
        input_dim=input_dim,
        hidden_dim=hidden_dim,
        bottleneck_dim=bottleneck_dim,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for epoch in range(num_epochs):

      if epoch in {20, 30, 35}:
        lr = lr / 2
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

      epoch_loss = 0.0

      for (batch_x,) in loader:
        batch_x = batch_x.to(device)

        optimizer.zero_grad()
        recon = model(batch_x)
        loss = criterion(recon, batch_x)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_x.size(0)

      epoch_loss /= len(dataset)

      if verbose:
        name = dataset_name if dataset_name is not None else "<unknown>"
        print(f"[AE] {name} | epoch {epoch+1}/{num_epochs} | loss={epoch_loss:.6f}")

    # Reconstruction error for each window
    model.eval()
    with torch.no_grad():
        X_tensor = torch.from_numpy(X).to(device)
        recon_all = model(X_tensor)
        errors = ((recon_all - X_tensor) ** 2).mean(dim=1).cpu().numpy()

    # Map window level errors back to full time index
    recon_error = errors
    full_scores = np.zeros(n, dtype=float)
    full_scores[:] = np.nan

    offset = window_size - 1
    for i, err in enumerate(recon_error):
        t_idx = offset + i
        full_scores[t_idx] = err

    first_valid = np.nanmin(full_scores)
    full_scores[:offset] = first_valid

    scores = pd.Series(
        full_scores,
        index=df.index,
        name="ae_recon_error",
    )

    return scores, model, scaler

def run_autoencoder_on_all_datasets(
    processed: dict,
    value_col: str = "value",
    window_size: int = 20,
    hidden_dim: int = 32,
    bottleneck_dim: int = 8,
    num_epochs: int = 10,
    batch_size: int = 128,
    lr: float = 1e-3,
):
    ae_scores = {}
    ae_models = {}
    ae_scalers = {}

    # for debugging you can keep this slice, then remove to run all:
    items = list(processed.items())  # [:5]
    total = len(items)

    import time

    for i, (key, df) in enumerate(items, start=1):
        start = time.time()
        print(f"\n[AE] Dataset {i}/{total}: {key}")

        scores, model, scaler = fit_autoencoder_for_dataset(
            df,
            value_col=value_col,
            window_size=window_size,
            hidden_dim=hidden_dim,
            bottleneck_dim=bottleneck_dim,
            num_epochs=num_epochs,
            batch_size=batch_size,
            lr=lr,
            dataset_name=key,
            verbose=True,
        )

        print(
            f"[AE] Completed dataset {i}/{total}: {key} "
            f"(elapsed {time.time() - start:.2f}s)"
        )

        ae_scores[key] = scores
        ae_models[key] = model
        ae_scalers[key] = scaler

    return ae_scores, ae_models, ae_scalers

## Run AE + evaluate + save ##
ae_scores, ae_models, ae_scalers = run_autoencoder_on_all_datasets(
    processed,
    value_col="value",
    window_size=20,
    hidden_dim=32,
    bottleneck_dim=8,
    num_epochs=40,
    batch_size=128,
    lr=1e-3,
)

print(list(ae_scores.keys())[:5])
print(ae_scores[list(ae_scores.keys())[0]].head())

ae_perf = evaluate_isolation_forest(
    iso_scores=ae_scores,
    processed=processed,
    top_frac=0.01,
)

print(ae_perf.head())
print(ae_perf.describe())

# Save AE performance scores for use back on laptop: export from google drive to laptop
ae_perf.to_csv(ROOT / "ae_performance.csv", index_label="dataset")

Mounted at /content/drive
Loaded datasets: 58
Labelled files: 58
Processed datasets: 58

['realKnownCause/rogue_agent_key_hold', 'realKnownCause/ambient_temperature_system_failure', 'realKnownCause/machine_temperature_system_failure', 'realKnownCause/nyc_taxi', 'realKnownCause/ec2_request_latency_system_failure']
0    0.000079
1    0.000079
2    0.000079
3    0.000079
4    0.000079
Name: ae_recon_error, dtype: float64
                                                   precision    recall  \
dataset                                                                  
realKnownCause/rogue_agent_key_hold                 0.000000  0.000000   
realKnownCause/ambient_temperature_system_failure   0.000000  0.000000   
realKnownCause/machine_temperature_system_failure   0.277533  0.027778   
realKnownCause/nyc_taxi                             0.394231  0.039614   
realKnownCause/ec2_request_latency_system_failure   1.000000  0.118497   

                                                         f1

In [5]:
print("Average Autoencoder F1:", ae_perf["f1"].mean())

print("Variance of Autoencoder F1:", ae_perf["f1"].var())


Average Autoencoder F1: 0.08416318767121776
Variance of Autoencoder F1: 0.0045336521103968895
